<a href="https://colab.research.google.com/github/google-research/skai/blob/skai-colab/Initialize_SKAI_Colab_Kernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialize SKAI Colab Kernel
This notebook contains a set of commands to run on a new custom colab kernel VM. These commands will pre-install all the necessary dependencies in order to run the SKAI pilot notebook.

To use:
1. **Open notebook in Playground mode. This is so that you don't accidentally save settings changes you make.**
2. Connect to the custom kernel you want to initialize.
3. Change settings as appropriate.
4. Run all code cells.


In [ ]:
#@title Settings
CLOUD_PROJECT_NAME = "" #@param {type:"string"}
CLOUD_LOCATION = "europe-west1" #@param {type:"string"}
CLOUD_SERVICE_ACCOUNT = "" #@param {type:"string"}
PRIVATE_KEY_PATH = "/root/service-account-private-key.json" #@param {type:"string"}
VIRTUALENV_DIR = "/content/skai-env" #@param {type:"string"}
SKAI_REPO = "https://github.com/google-research/skai.git"  #@param {type:"string"}
SKAI_CODE_DIR = "/content/skai-src"  #@param {type:"string"}

# Install newer version of GDAL
The default GDAL library version installed on custom colab kernels is out of date. These commands will update to a newer version.

In [ ]:
%%shell
sudo add-apt-repository -y ppa:ubuntugis/ubuntugis-unstable
sudo apt-get update
sudo apt-get install gdal-bin
gdalinfo --version | awk '$2 ~ /^3\./ {exit 0} {exit 1}'

# Install Python libraries for kernel
These are dependencies needed by the colab notebook itself. This is distinct from installing the python dependencies into the SKAI virtualenv.

In [ ]:
%shell pip --use-deprecated=legacy-resolver install google-cloud-aiplatform appengine-python-standard ipyplot google-cloud-monitoring pyproj

# Create service account key file
Because the custom kernel VM is shared among multiple users, users should not authenticate with GCP using their own accounts. Otherwise, their private credentials will be written to the kernel's filesystem for all other users to see.

Instead, the colab should authenticate as a service account. In order to do that, a private key for the service account should be created and uploaded to the kernel. This should be done either via Cloud's web console following [these instructions](https://cloud.google.com/iam/docs/creating-managing-service-account-keys#iam-service-account-keys-create-console), or the user can run `gcloud` on their workstation to create the key file. The key file should end with ".json".

Regardless of how the key is created, it should be uploaded to the kernel.

**Note 1:** For security, a new private key should be created for each kernel, instead of reusing one key for multiple kernels. This way, we can revoke a kernel's permissions by deleting the key.

**Note 2:** Currently, newly created private keys never expire. We probably want to set an expiry date for the key of a couple of months for security. See [here](https://cloud.google.com/iam/docs/service-accounts#key-expiry) for details.

In [ ]:
from google.colab import files

print('Use Cloud web console to create a service account private key, or run the following command on your workstation:\n')
print(f'gcloud iam service-accounts keys create service-account-private-key.json --iam-account={CLOUD_SERVICE_ACCOUNT}\n')
print('Then upload the created file to to colab')

uploaded = files.upload()
if len(uploaded) == 1:
  KEY_FILE = list(uploaded.keys())[0]
  !mv {KEY_FILE} {PRIVATE_KEY_PATH}


# Set Cloud project and service account
1. Set project name and default location.
2. Activate the service account.

In [ ]:
%shell gcloud config set project {CLOUD_PROJECT_NAME}
%shell gcloud auth activate-service-account {CLOUD_SERVICE_ACCOUNT} --key-file={PRIVATE_KEY_PATH}
%shell gcloud config set compute/region {CLOUD_LOCATION}


# Clone SKAI github repo

In [ ]:
%shell rm -rf {SKAI_CODE_DIR}
%shell git clone {SKAI_REPO} {SKAI_CODE_DIR}

# Create Python virtualenv for running SKAI
1. Install virtualenv module using pip.
2. Create "skai-env" virtual environment.
3. Install Python dependencies into virtual environment using SKAI's requirements.txt

In [ ]:
%%bash -s "$VIRTUALENV_DIR" "$SKAI_CODE_DIR"
set -e

pip3 install virtualenv
virtualenv "$1"
source "$1/bin/activate"
pip install -r "$2/requirements.txt"